# Phase 4 — Spec-compliant real-model pipeline

Qwen2.5-1.5B-Instruct + calibrated W_OLS + context 4096. Chạy từng cell và kiểm tra output trước khi sang cell tiếp theo.

In [ ]:
# Cell 1 — setup repository/dependencies
!nvidia-smi
!pip install -q transformers accelerate matplotlib seaborn pandas
import os, sys, subprocess, gc, json, time, random, datetime
from pathlib import Path
ROOT=Path.cwd()
if not (ROOT/'src').exists():
    ROOT=ROOT/'vector-kv-cache'
    if not (ROOT/'src').exists(): subprocess.run(['git','clone','--depth','1','https://github.com/Dung34/vector-kv-cache.git',str(ROOT)],check=True)
for name in list(sys.modules):
    if name=='src' or name.startswith('src.'): del sys.modules[name]
sys.path.insert(0,str(ROOT))
print('ROOT=',ROOT)

In [ ]:
# Cell 2 — imports, device and reproducibility
import torch, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from transformers import AutoTokenizer, AutoModelForCausalLM
from src.baselines.full_cache import FullKVCacheWrapper
from src.baselines.h2o_cache import H2OKVCache
from src.baselines.h2o_hook import H2OAttentionHookManager
from src.vector_cache.vector_cache import VectorKVCache
from src.vector_cache.vector_hook import VectorAttentionHookManager
SEED=42; random.seed(SEED); torch.manual_seed(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); assert DEVICE.type=='cuda'
DTYPE=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print({'gpu':torch.cuda.get_device_name(0),'dtype':str(DTYPE),'seed':SEED})

In [ ]:
# Cell 3 — load Qwen2.5-1.5B-Instruct
MODEL_ID='Qwen/Qwen2.5-1.5B-Instruct'
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,trust_remote_code=True)
if tokenizer.pad_token is None: tokenizer.pad_token=tokenizer.eos_token
model=AutoModelForCausalLM.from_pretrained(MODEL_ID,torch_dtype=DTYPE,device_map='auto',attn_implementation='eager',trust_remote_code=True).eval()
print({'model':MODEL_ID,'layers':model.config.num_hidden_layers,'hidden_size':model.config.hidden_size,'kv_heads':model.config.num_key_value_heads})

In [ ]:
# Cell 4 — load and validate calibrated W_OLS
candidate_paths=[ROOT/'weights/w_ols_qwen2.5_1.5b.pt',Path('weights/w_ols_qwen2.5_1.5b.pt'),Path('w_ols_qwen2.5_1.5b.pt')]
weights_path=next((p for p in candidate_paths if p.exists()),None); assert weights_path is not None, 'Missing calibrated W_OLS checkpoint'
loaded=torch.load(weights_path,map_location='cpu')
w_ols=loaded['w_ols'] if isinstance(loaded,dict) and 'w_ols' in loaded else loaded
expected=(model.config.num_hidden_layers,model.config.num_key_value_heads,model.config.hidden_size//model.config.num_attention_heads,model.config.hidden_size//model.config.num_attention_heads)
print({'weights_path':str(weights_path),'actual_shape':tuple(w_ols.shape),'expected_shape':expected})
assert tuple(w_ols.shape)==expected, 'W_OLS shape does not match the loaded model'
w_ols=w_ols.to(device=DEVICE,dtype=DTYPE)

In [ ]:
# Cell 5 — build a real 4096-token prompt
CONTEXT_LENGTH=4096; NEW_TOKENS=64
text=('Long-context inference requires careful memory accounting and reliable retrieval. ' * 1000)
ids=tokenizer(text,return_tensors='pt',add_special_tokens=False).input_ids[:,:CONTEXT_LENGTH].to(DEVICE)
assert ids.shape[-1]==CONTEXT_LENGTH
print({'prompt_tokens':ids.shape[-1],'new_tokens':NEW_TOKENS})

In [ ]:
# Cell 6 — benchmark one configuration
def run_one(name,pc,pa=0.0):
    manager=None; cache=None
    if name=='full': cache=FullKVCacheWrapper()
    elif name=='h2o': cache=H2OKVCache(compression_ratio=pc,num_sink_tokens=4) ; manager=H2OAttentionHookManager(model,cache)
    else: cache=VectorKVCache(compression_ratio=pc,approx_ratio=pa,num_sink_tokens=4,w_ols=w_ols); manager=VectorAttentionHookManager(model,cache)
    if manager: manager.register_hooks()
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize()
    with torch.inference_mode():
        t0=time.perf_counter(); out=model(input_ids=ids,past_key_values=cache,use_cache=True); torch.cuda.synchronize()
    prefill_ms=(time.perf_counter()-t0)*1000; generated=[]; times=[]
    for _ in range(NEW_TOKENS):
        token=out.logits[:,-1:].argmax(-1); generated.append(token.item()); torch.cuda.synchronize(); t0=time.perf_counter()
        with torch.inference_mode(): out=model(token,past_key_values=cache,use_cache=True)
        torch.cuda.synchronize(); times.append(time.perf_counter()-t0)
    if manager: manager.remove_hooks()
    result={'name':name,'pc':pc,'pa':pa,'prefill_ms':prefill_ms,'decode_tok_s':NEW_TOKENS/sum(times),'itl_ms':sum(times)/len(times)*1000,'persistent_kv_bytes':cache.get_memory_bytes(),'peak_allocated':torch.cuda.max_memory_allocated(),'peak_reserved':torch.cuda.max_memory_reserved(),'physical_len':cache._physical_len(0) if hasattr(cache,'_physical_len') else cache.get_seq_length(0),'generated':generated}
    print(result); return result

In [ ]:
# Cell 7 — run all spec budgets
configs=[('full',0.0,0.0),('h2o',0.50,0.0),('vector',0.50,0.20),('h2o',0.75,0.0),('vector',0.75,0.20),('h2o',0.85,0.0),('vector',0.85,0.20)]
results=[run_one(*config) for config in configs]
df=pd.DataFrame([{k:v for k,v in r.items() if k!='generated'} for r in results]); df

In [ ]:
# Cell 8 — charts and token comparison
sns.set_theme(style='whitegrid'); fig,ax=plt.subplots(1,3,figsize=(20,5))
sns.barplot(data=df,x='pc',y='prefill_ms',hue='name',ax=ax[0]); ax[0].set_title('Prefill latency')
sns.barplot(data=df,x='pc',y='decode_tok_s',hue='name',ax=ax[1]); ax[1].set_title('Decode throughput')
sns.barplot(data=df,x='pc',y='persistent_kv_bytes',hue='name',ax=ax[2]); ax[2].set_title('Persistent KV bytes')
plt.tight_layout(); plt.show()
baseline=results[0]['generated']
for item in results[1:]: print(item['name'],item['pc'],{'first_token':item['generated'][0]==baseline[0],'exact_sequence':item['generated']==baseline})

In [ ]:
# Cell 9 — save raw report
report_dir=ROOT/'reports'; report_dir.mkdir(exist_ok=True)
with open(report_dir/'phase4_spec_4096.json','w') as f: json.dump({'model':MODEL_ID,'context_length':CONTEXT_LENGTH,'new_tokens':NEW_TOKENS,'seed':SEED,'results':results},f,indent=2)
df.to_markdown(report_dir/'phase4_spec_4096.md',index=False)
print('reports:',report_dir/'phase4_spec_4096.json',report_dir/'phase4_spec_4096.md')

## NIAH cells

Sau khi Cell 7 chạy ổn định, bổ sung/chạy evaluator single-needle và multi-needle. Không kết luận recall từ benchmark latency; NIAH phải có heatmap theo context length và needle depth.